# Importing The Libraries


In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import random

# Loading All The Datasets

In [4]:
superStoreDf = pd.read_csv('datasets/Superstore Dataset.csv', encoding = 'latin1')
demandDf = pd.read_csv('datasets/demand_forecasting.csv')
inventoryDf = pd.read_csv('datasets/inventory_monitoring.csv')
priceDf = pd.read_csv('datasets/pricing_optimization.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'datasets/Superstore Dataset.csv'

Getting The Size of The Dataset

In [ ]:
print(superStoreDf.shape)
print(demandDf.shape)
print(inventoryDf.shape)
print(priceDf.shape)

Getting All The Columns

In [ ]:
print(superStoreDf.columns)
print(demandDf.columns)
print(inventoryDf.columns)
print(priceDf.columns)

**Observations**
- We really don't need the Superstore Dataset because we can get the all the important attributes from the three datasets only.
- So I will work on the three datasets only.

# Merging The Datasets
1. Checking The Datatypes of all the Three Dataset for merging

In [ ]:
print(demandDf.dtypes)
print(inventoryDf.dtypes)
print(priceDf.dtypes)

**Observations**
- As, two columns "StoreID" and "ProductID" is same in the three datasets and having the same datatype of integer we will use this to merge the datasets.

# Merging The Demand and Inventory Dataset

In [ ]:
# mergedDf = pd.merge(
#     demandDf,
#     inventoryDf,
#     on = ['Product ID', 'Store ID'],
#     how = 'inner'
# )


**Observations**
- When I use this and use the two columns i.e Product ID and Store ID the dataset size significantly reduce to the 122 records because sometimes it happens that Store ID not matched while Product ID matched so I have to try on the Product ID only

In [ ]:
mergedDf = pd.merge(
    demandDf,
    inventoryDf,
    on = ['Product ID'],
    how = 'inner'
)

In [ ]:
mergedDf.shape
# mergedDf.head().T


**Observations**
- As we can see that the Store ID x and Store ID y have been created because it may possible that the product exists in multiple stores.
- We need to get rid of it in our next section.
- But first, merge the price dataset with it.


# Merging The Price Dataset with Merged Dataset.

In [ ]:
mergedDf = pd.merge(
    mergedDf,
    priceDf,
    on = ['Product ID'],
    how = 'inner',
    suffixes = ('', '_pricing')
)

In [ ]:
mergedDf.shape
# mergedDf.head(1).T

In [ ]:
mergedDf.isnull().sum()

# Finding The Unique Product IDs for imputing it with the Synthetic Data.


In [ ]:
mergedDf['Product ID'].value_counts()

# Renaming The Columns For Store ID

In [ ]:
mergedDf.rename(columns={

    'Store ID_x': 'Online Store ID',
    'Store ID_y': 'Warehouse ID',
    'Store ID': 'Retail Store ID',

    'Price': 'Selling Price',
    'Price_pricing': 'Market Price'

}, inplace=True)

mergedDf.head().T

# Creating The Synthetic Data For Product ID and Store ID

In [ ]:
product_categories = {

    "Electronics": [
        "Laptop",
        "Smartphone",
        "Keyboard",
        "Monitor",
        "Headphones"
    ],

    "Fashion": [
        "T-Shirt",
        "Jeans",
        "Sneakers",
        "Jacket",
        "Hoodie"
    ],

    "Furniture": [
        "Chair",
        "Desk",
        "Table",
        "Sofa",
        "Bookshelf"
    ],

    "Groceries": [
        "Rice",
        "Milk",
        "Coffee",
        "Snacks",
        "Bread"
    ],

    "Home Appliances": [
        "Microwave",
        "Fan",
        "Mixer",
        "Refrigerator",
        "Washing Machine"
    ]
}


# Imputing It With The Product ID
- For this, I will create the product mapping which will helps us to impute the product name with product IDs

In [ ]:
product_mapping = {}

unique_products = mergedDf[
    'Product ID'
].unique()

for product_id in unique_products:

    category = random.choice(
        list(product_categories.keys())
    )

    product_name = random.choice(
        product_categories[category]
    )

    product_mapping[product_id] = {

        'Product Name': product_name,
        'Category': category

    }

# Mapping The Product Name and Category with appropriate Product ID

In [ ]:
mergedDf['Product Name'] = mergedDf[
    'Product ID'
].map(

    lambda x:
    product_mapping[x]['Product Name']

)

mergedDf['Category'] = mergedDf[
    'Product ID'
].map(

    lambda x:
    product_mapping[x]['Category']

)

In [ ]:
mergedDf.head(1).T

# Imputing The Store IDs with proper Synthetic Store Name

In [ ]:
online_store_names = [

    "QuickCart Online",
    "ShopEase Online",
    "MegaBuy Online",
    "UrbanCart",
    "SmartBasket"

]

warehouse_names = [

    "Central Warehouse",
    "North Distribution Hub",
    "Prime Storage",
    "Logistics Hub",
    "Supply Chain Center"

]

retail_store_names = [

    "City Retail Mart",
    "Retail Express",
    "Downtown Store",
    "Value Retail Store",
    "SuperMart Outlet"

]

# Creating The Locations Also To Track Which city have the most revenue
locations = [

    "Ahmedabad",
    "Mumbai",
    "Delhi",
    "Pune",
    "Bangalore",
    "Hyderabad",
    "Chennai",
    "Kolkata"

]


In [ ]:
print(
    mergedDf['Online Store ID'].value_counts()
)
print(
    mergedDf['Online Store ID'].value_counts()
)
print(
    mergedDf['Warehouse ID'].value_counts()
)

# Creating The Same Mapping for it also

In [ ]:
# For Online Store IDs
online_mapping = {}

for store_id in mergedDf[
    'Online Store ID'
].dropna().unique():

    online_mapping[store_id] = {

        'Store Name':
            random.choice(online_store_names),

        'Location':
            random.choice(locations)

    }

# For Warehouse Mapping

warehouseMapping = {}

for storeId in mergedDf[
    'Warehouse ID'
].dropna().unique():

    warehouseMapping[storeId] = {

        'Store Name':
            random.choice(warehouse_names),

        'Location':
            random.choice(locations)

    }

# For Retail Store Mapping
retailMapping = {}

for storeId in mergedDf[
    'Retail Store ID'
].dropna().unique():

    retailMapping[storeId] = {

        'Store Name':
            random.choice(retail_store_names),

        'Location':
            random.choice(locations)

    }


# Map The Store IDs

In [ ]:
# Mapping Online Store ID
mergedDf['Online Store Name'] = mergedDf[
    'Online Store ID'
].map(

    lambda x:
    online_mapping[x]['Store Name']

)

mergedDf['Online Store Location'] = mergedDf[
    'Online Store ID'
].map(

    lambda x:
    online_mapping[x]['Location']

)

# Mapping Warehouse Store IDs
mergedDf['Warehouse Name'] = mergedDf[
    'Warehouse ID'
].map(

    lambda x:
    warehouseMapping[x]['Store Name']

)

mergedDf['Warehouse Location'] = mergedDf[
    'Warehouse ID'
].map(

    lambda x:
    warehouseMapping[x]['Location']

)

# Mapping Retail Store ID
mergedDf['Retail Store Name'] = mergedDf[
    'Retail Store ID'
].map(

    lambda x:
    retailMapping[x]['Store Name']

)

mergedDf['Retail Store Location'] = mergedDf[
    'Retail Store ID'
].map(

    lambda x:
    retailMapping[x]['Location']

)

# Checking The Final Dataset

In [ ]:
mergedDf.head(1).T

# Creating The Final Dataset With Appropriate Columns

In [ ]:
final_columns = [
    'Product ID',
    'Product Name',
    'Category',
    'Date',
    'Online Store Location',
    'Sales Quantity',
    'Selling Price',
    'Promotions',
    'Demand Trend',
    'Customer Segments',
    'Warehouse Location',
    'Stock Levels',
    'Stockout Frequency',
    'Reorder Point',
    'Order Fulfillment Time (days)',
    'Retail Store Location',
    'Market Price',
    'Competitor Prices',
    'Discounts',
    'Sales Volume',
    'Return Rate (%)',
    'Elasticity Index'

]

In [ ]:
final_columns = [
    col for col in final_columns
    if col in mergedDf.columns
]


In [ ]:
finalDataset = mergedDf[
    final_columns
]

Checking The Shape, Columns and Preview

In [ ]:
print("\nFinal Dataset Shape:")
print(finalDataset.shape)

print("\nFinal Columns:\n")
print(finalDataset.columns.tolist())


# Save The Final Dataset

In [ ]:
finalDataset.to_csv('Omnichannel Retail Data.csv', index = False)
print("Dataset Saved Successfully")